# Taipei Shelter-Road Analysis

## stage 1 - pre
---

In [2]:
import os, json, ast
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from dotenv import load_dotenv

load_dotenv()
os.makedirs("output", exist_ok=True)


In [3]:
place_name = "臺北市, 臺灣"
graph_path = "output/taipei_drive.graphml"

if os.path.exists(graph_path):
    G = ox.load_graphml(graph_path)
else:
    G = ox.graph_from_place(place_name, network_type="drive_service", simplify=True)

G = ox.project_graph(G, to_crs="EPSG:3826")
print("Road graph loaded. Nodes:", len(G.nodes), "Edges:", len(G.edges))


Road graph loaded. Nodes: 31582 Edges: 74509


In [4]:
DEFAULT_SPEEDS = json.loads(os.getenv("DEFAULT_SPEEDS", "{}"))
if not DEFAULT_SPEEDS:
    DEFAULT_SPEEDS = {
        "motorway": 90, "motorway_link": 50, "trunk": 70, "trunk_link": 50,
        "primary": 60, "primary_link": 50, "secondary": 50, "secondary_link": 40,
        "tertiary": 40, "tertiary_link": 40, "residential": 30, "unclassified": 30,
        "living_street": 20, "service": 20, "busway": 40
    }

def pick_first(val):
    if isinstance(val, list):
        return val[0] if val else None
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = ast.literal_eval(s)
            if isinstance(arr, list) and arr:
                return arr[0]
        except Exception:
            pass
    return s

def to_float_or_none(x):
    try:
        return float(x)
    except Exception:
        return None

def classify_road(data):
    hw = str(pick_first(data.get("highway")) or "").lower()
    bridge = str(pick_first(data.get("bridge")) or "no").lower()
    tunnel = str(pick_first(data.get("tunnel")) or "no").lower()
    if hw in {"motorway", "motorway_link", "trunk", "trunk_link"}:
        return "expressway"
    if bridge in {"yes", "viaduct"}:
        return "bridge"
    if tunnel in {"yes", "building_passage"}:
        return "underground_road"
    if hw == "service":
        return "service"
    if hw in {"primary", "primary_link", "secondary", "secondary_link"}:
        return "arterial"
    return "residential"

for _, _, _, data in G.edges(keys=True, data=True):
    hw = str(pick_first(data.get("highway")) or "").lower()
    spd = to_float_or_none(data.get("speed_kph"))
    if spd is None and hw in DEFAULT_SPEEDS:
        spd = float(DEFAULT_SPEEDS[hw])
    if spd is not None and str(pick_first(data.get("bridge")) or "no").lower() in {"yes", "viaduct"}:
        spd += 10.0
    if spd is not None:
        data["speed_kph"] = spd
        length_m = to_float_or_none(data.get("length"))
        if length_m is not None and spd > 0:
            data["travel_time"] = length_m / (spd * 1000 / 3600)
    data["road_type"] = classify_road(data)

ox.save_graphml(G, graph_path)
print("Saved:", graph_path)


Saved: output/taipei_drive.graphml


## shelters sjoin
---

In [5]:
# shelters 清理 + 台北市篩選 + 轉 GDF + 500m 內最近道路配對（合併版）

import pandas as pd
import geopandas as gpd
import osmnx as ox

# 1) 讀取與清理 shelters
shelters = pd.read_csv("data/shelter_marked.csv")
shelters.columns = [c.strip().strip("'").strip('"') for c in shelters.columns]

for c in shelters.columns:
    if shelters[c].dtype == "object":
        shelters[c] = shelters[c].astype(str).str.strip().str.strip("'").str.strip('"')

# 2) 篩選台北市
if "COUNTYNAME" in shelters.columns:
    mask_tp = shelters["COUNTYNAME"].str.contains("臺北市|台北市", na=False, regex=True)
else:
    mask_tp = shelters["縣市及鄉鎮市區"].str.contains("臺北市|台北市", na=False, regex=True)

shelters_tp = shelters[mask_tp].copy()

# 3) 轉 GeoDataFrame（WGS84）
lon_col, lat_col = "經度", "緯度"
shelters_tp_gdf = gpd.GeoDataFrame(
    shelters_tp,
    geometry=gpd.points_from_xy(
        pd.to_numeric(shelters_tp[lon_col], errors="coerce"),
        pd.to_numeric(shelters_tp[lat_col], errors="coerce"),
    ),
    crs="EPSG:4326",
).dropna(subset=["geometry"])

# 4) 轉路網 edges，CRS 對齊
edges_gdf = ox.graph_to_gdfs(G, nodes=False).copy()
shelters_proj = shelters_tp_gdf.to_crs(edges_gdf.crs) if shelters_tp_gdf.crs != edges_gdf.crs else shelters_tp_gdf.copy()

# 5) 每個 shelter 建唯一 ID
shelters_proj = shelters_proj.reset_index(drop=True)
shelters_proj["shelter_id"] = shelters_proj.index

# 6) 500m 最近道路 join
use_cols = [c for c in ["highway", "road_type", "length", "speed_kph", "travel_time", "geometry"] if c in edges_gdf.columns]
shelter_join = gpd.sjoin_nearest(
    shelters_proj,
    edges_gdf[use_cols],
    how="left",
    max_distance=500,
    distance_col="dist_to_edge_m"
)

# 7) 只留 500m 內有道路，且每個 shelter 只取最近一筆
matched_unique = (
    shelter_join[shelter_join["dist_to_edge_m"].notna()]
    .sort_values(["shelter_id", "dist_to_edge_m"])
    .drop_duplicates(subset=["shelter_id"], keep="first")
    .copy()
)

# 8) 輸出
matched_unique.to_file("output/taipei_shelters_join_500m.geojson", driver="GeoJSON", encoding="utf-8")

print("原 shelter 數:", len(shelters_proj))
print("保留 shelter 數 (500m 內有道路):", len(matched_unique))
print("移除 shelter 數:", len(shelters_proj) - len(matched_unique))

原 shelter 數: 311
保留 shelter 數 (500m 內有道路): 311
移除 shelter 數: 0


## Grid establishment
---

In [6]:
# Cell 1: import
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
from shapely.geometry import box


In [7]:
# Cell 2: 建立台北市 250m x 250m grid（EPSG:3826）
town_path = "data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp"
town = gpd.read_file(town_path)
town.columns = [c.strip().strip("'").strip('"') for c in town.columns]

tp = town[town["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)].copy()
tp = tp.to_crs(epsg=3826)
tp_boundary = tp.dissolve()

minx, miny, maxx, maxy = tp_boundary.total_bounds
cell = 250

xs = np.arange(minx, maxx, cell)
ys = np.arange(miny, maxy, cell)
grid = gpd.GeoDataFrame({"geometry": [box(x, y, x+cell, y+cell) for x in xs for y in ys]}, crs="EPSG:3826")

grid_tp = gpd.overlay(grid, tp_boundary[["geometry"]], how="intersection")
grid_tp["grid_id"] = np.arange(1, len(grid_tp) + 1)

print("grid count:", len(grid_tp))


grid count: 4585


## 地形疊圖grid分析

In [8]:
# Cell 3: DEM -> 每格 mean_elevation + slope（degree）
dem_path = "data/Taipei_dem.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1).astype("float32")
    dem_nodata = src.nodata
    dem_transform = src.transform
    dem_crs = src.crs

if dem_crs != grid_tp.crs:
    with rasterio.open(dem_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, grid_tp.crs, src.width, src.height, *src.bounds
        )
        dst = np.empty((height, width), dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=grid_tp.crs,
            resampling=Resampling.bilinear
        )
        dem = dst
        dem_transform = transform
        dem_nodata = src.nodata

if dem_nodata is not None:
    dem[dem == dem_nodata] = np.nan

cellsize_x = abs(dem_transform.a)
cellsize_y = abs(dem_transform.e)
gy, gx = np.gradient(dem, cellsize_y, cellsize_x)
slope_raster = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2))).astype("float32")

elev_stats = zonal_stats(grid_tp.geometry, dem, affine=dem_transform, stats=["mean"], nodata=np.nan)
slope_stats = zonal_stats(grid_tp.geometry, slope_raster, affine=dem_transform, stats=["mean"], nodata=np.nan)

grid_final = grid_tp.copy()
grid_final["mean_elevation"] = [s["mean"] for s in elev_stats]
grid_final["slope"] = [s["mean"] for s in slope_stats]

print("elevation non-null:", grid_final["mean_elevation"].notna().sum(), "/", len(grid_final))
print("slope non-null:", grid_final["slope"].notna().sum(), "/", len(grid_final))


elevation non-null: 4558 / 4585
slope non-null: 4558 / 4585


## 河川疊圖grid分析

In [9]:
# Cell 4: 河川疊圖 -> river_ratio, dist_to_river_m
river_path = "data/riverpoly/riverpoly.shp"
river = gpd.read_file(river_path)
river_3826 = river.to_crs(epsg=3826)

grid_river = grid_tp.to_crs(epsg=3826).copy()
grid_river["grid_area_m2"] = grid_river.geometry.area

inter = gpd.overlay(
    grid_river[["grid_id", "grid_area_m2", "geometry"]],
    river_3826[["geometry"]],
    how="intersection"
)

if len(inter) > 0:
    inter["river_area_m2"] = inter.geometry.area
    river_area = inter.groupby("grid_id", as_index=False)["river_area_m2"].sum()
else:
    river_area = pd.DataFrame({"grid_id": [], "river_area_m2": []})

grid_river = grid_river.merge(river_area, on="grid_id", how="left")
grid_river["river_area_m2"] = grid_river["river_area_m2"].fillna(0.0)
grid_river["river_ratio"] = (grid_river["river_area_m2"] / grid_river["grid_area_m2"]).fillna(0.0)
grid_river["has_river"] = grid_river["river_area_m2"] > 0

grid_river["dist_to_river_m"] = 0.0
need_dist = grid_river["river_ratio"] == 0
if need_dist.any():
    targets = grid_river.loc[need_dist, ["grid_id", "geometry"]].copy()
    nn = gpd.sjoin_nearest(targets, river_3826[["geometry"]], how="left", distance_col="dist_to_river_m")
    dist_map = nn.groupby("grid_id")["dist_to_river_m"].min()
    grid_river.loc[need_dist, "dist_to_river_m"] = grid_river.loc[need_dist, "grid_id"].map(dist_map)

print("has river grids:", int(grid_river["has_river"].sum()))


has river grids: 851


## 讀取真實淹水資料疊圖grid分析

In [10]:
# 讀取淹水資料 + 轉 EPSG:3826 + 依日期分組 + sjoin 到 grid + 計算每格發生過幾天淹水

import geopandas as gpd
import pandas as pd

flood_path = "data/淹水.gpkg"
flood_layer = "111111111111__1141019__1140307"

# 1) 讀取淹水 GeoPackage，原始 CRS 指定為 EPSG:4326，轉成 EPSG:3826
flood_raw = gpd.read_file(flood_path, layer=flood_layer)
flood_raw = flood_raw.set_crs(epsg=4326, allow_override=True)

flood_3826 = flood_raw.to_crs(epsg=3826)

# 2) 處理日期欄位
flood_3826["flood_date"] = pd.to_datetime(
    flood_3826["FDATE"],
    format="%Y/%m/%d",
    errors="coerce"
)
flood_3826["flood_date_str"] = flood_3826["flood_date"].dt.strftime("%Y-%m-%d")

# 3) 依不同日期分出淹水資料
# key = 日期字串，例如 "2024-07-10"
# value = 該日期的 GeoDataFrame
flood_by_date = {
    date: group.copy()
    for date, group in flood_3826.groupby("flood_date_str")
    if pd.notna(date)
}

# 4) 每個日期有幾筆淹水 polygon
flood_date_count = (
    flood_3826.groupby("flood_date_str")
    .size()
    .reset_index(name="count")
    .sort_values("flood_date_str")
)

# 5) 準備 grid
grid_for_flood = grid_final[["grid_id", "geometry"]].copy().to_crs(epsg=3826)

# 6) Spatial join：每個 grid cell 跟哪些淹水 polygon 相交
grid_flood_sjoin = gpd.sjoin(
    grid_for_flood,
    flood_3826[["flood_date_str", "geometry"]],
    how="left",
    predicate="intersects"
)

# 7) 每個 grid cell 出現過幾個不同淹水日期
flood_event_count = (
    grid_flood_sjoin
    .dropna(subset=["flood_date_str"])
    .groupby("grid_id")["flood_date_str"]
    .nunique()
    .reset_index(name="flood_event_count")
)

# 8) 合併回原本的 g
g = grid_final.drop(columns=["flood_event_count"], errors="ignore")
g = grid_final.merge(flood_event_count, on="grid_id", how="left")
g["flood_event_count"] = g["flood_event_count"].fillna(0).astype(int)

print("淹水資料原始 CRS: EPSG:4326")
print("淹水資料轉換後 CRS:", flood_3826.crs)
print("淹水資料總筆數:", len(flood_3826))
print("淹水日期數:", len(flood_by_date))
print("有淹水紀錄的 grid 數:", (g["flood_event_count"] > 0).sum())
print("最大淹水日期次數:", g["flood_event_count"].max())

display(flood_date_count)

display(
    g[["grid_id", "flood_event_count"]]
    .sort_values("flood_event_count", ascending=False)
    .head(20)
)

淹水資料原始 CRS: EPSG:4326
淹水資料轉換後 CRS: EPSG:3826
淹水資料總筆數: 317
淹水日期數: 51
有淹水紀錄的 grid 數: 309
最大淹水日期次數: 11


,flood_date_str,count
0,2008-05-27,1
1,2008-09-12,8
2,2008-09-27,4
3,2009-08-12,4
4,2010-06-21,1
5,2011-06-28,2
6,2012-04-22,8
7,2012-05-02,4
8,2012-06-12,14
9,2012-06-16,9


,grid_id,flood_event_count
1070,1071,11
1069,1070,9
166,167,7
1239,1240,7
1320,1321,6
1821,1822,5
485,486,5
486,487,5
1820,1821,5
1238,1239,5


## 風險分數評估
---
- 地勢風險
- 河川風險
- 歷史淹水記錄風險
- 邊坡風險

In [11]:
# Cell 5: 四項風險指標（地形淹水 / 河川淹水 / 歷史淹水 / 邊坡）

g = grid_final.copy()

# 合併河川資訊
g = g.merge(
    grid_river[["grid_id", "river_ratio", "dist_to_river_m", "has_river", "river_area_m2", "grid_area_m2"]],
    on="grid_id",
    how="left"
)

g["river_ratio"] = g["river_ratio"].fillna(0.0)
g["dist_to_river_m"] = g["dist_to_river_m"].fillna(np.inf)
g["has_river"] = g["has_river"].fillna(False)

# 合併歷史淹水次數
# flood_event_count 是前面 sjoin 後算出來的表，欄位應該有 grid_id, flood_event_count
g = g.merge(
    flood_event_count,
    on="grid_id",
    how="left"
)

g["flood_event_count"] = g["flood_event_count"].fillna(0).astype(int)

# 高程分位數
q25 = g["mean_elevation"].quantile(0.25)
q50 = g["mean_elevation"].quantile(0.50)
q75 = g["mean_elevation"].quantile(0.75)

# 初始化四項指標
g["terrain_flood_risk"] = 0
g["river_flood_risk"] = 0
g["history_flood_risk"] = 0
g["slope_risk"] = 0

# 1) 地形淹水風險
g.loc[g["mean_elevation"] < q25, "terrain_flood_risk"] += 2
g.loc[
    (g["mean_elevation"] >= q25) & (g["mean_elevation"] < q50),
    "terrain_flood_risk"
] += 1

lowland = g["mean_elevation"] < q50

g.loc[
    lowland & (g["slope"] < 15),
    "terrain_flood_risk"
] += 1

# 2) 河川淹水風險
g.loc[
    g["has_river"] & (g["river_ratio"] > 0.5),
    "river_flood_risk"
] += 3

g.loc[
    g["has_river"] & (g["river_ratio"] > 0) & (g["river_ratio"] <= 0.5),
    "river_flood_risk"
] += 2

g.loc[
    (~g["has_river"]) & (g["dist_to_river_m"] < 500),
    "river_flood_risk"
] += 1

# 3) 歷史淹水風險
# 0 次      -> 0
# 1-3 次    -> 1
# 4-6 次    -> 2
# 7次以上    -> 3
g.loc[g["flood_event_count"].between(1, 3), "history_flood_risk"] = 1
g.loc[g["flood_event_count"].between(4, 6), "history_flood_risk"] = 2
g.loc[g["flood_event_count"] > 7, "history_flood_risk"] = 3

# 4) 邊坡風險
high_elev = g["mean_elevation"] > q75

g.loc[
    high_elev & (g["slope"] > 10) & (g["slope"] <= 20),
    "slope_risk"
] += 1

g.loc[
    high_elev & (g["slope"] > 20) & (g["slope"] <= 30),
    "slope_risk"
] += 2

g.loc[
    high_elev & (g["slope"] > 30),
    "slope_risk"
] += 3

# 淹水總風險：只加三個淹水相關指標，不含邊坡
g["total_flood_risk"] = (
    g["terrain_flood_risk"]
    + g["river_flood_risk"]
    + g["history_flood_risk"]
)

display(
    g[[
        "grid_id",
        "terrain_flood_risk",
        "river_flood_risk",
        "history_flood_risk",
        "total_flood_risk",
        "slope_risk"
    ]]
    .sort_values("total_flood_risk", ascending=False)
    .head(20)
)

,grid_id,terrain_flood_risk,river_flood_risk,history_flood_risk,total_flood_risk,slope_risk
1753,1754,3,3,1,7,0
1312,1313,3,3,1,7,0
1847,1848,3,3,1,7,0
485,486,3,2,2,7,0
165,166,3,2,2,7,0
484,485,3,2,2,7,0
486,487,3,2,2,7,0
0,1,3,3,0,6,0
6,7,3,3,0,6,0
4,5,3,3,0,6,0


In [12]:
# Cell 6: 輸出兩個檔案
# 1) GeoJSON（保留 geometry, EPSG:3826）
geojson_path = "output/Taipei_grid_full_risk.geojson"
g.to_file(geojson_path, driver="GeoJSON", encoding="utf-8")

# 2) CSV（改成經緯度欄位）
g_wgs84 = g.to_crs(epsg=4326).copy()

# 注意：因為 EPSG:4326 是經緯度座標，這裡只是用來輸出格網中心點 lon/lat
cent = g_wgs84.geometry.centroid
g_wgs84["lon"] = cent.x
g_wgs84["lat"] = cent.y

csv_cols = [
    "grid_id", "lon", "lat",
    "mean_elevation", "slope",
    "river_area_m2", "grid_area_m2", "river_ratio", "dist_to_river_m", "has_river",
    "terrain_flood_risk", "river_flood_risk", "history_flood_risk",
    "total_flood_risk", "slope_risk"
]

g_wgs84[csv_cols].to_csv(
    "output/Taipei_grid_full_risk.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", geojson_path)
print("Saved: output/Taipei_grid_full_risk.csv")

Saved: output/Taipei_grid_full_risk.geojson
Saved: output/Taipei_grid_full_risk.csv


C:\Users\Ben\AppData\Local\Temp\ipykernel_30912\3092559667.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = g_wgs84.geometry.centroid


## Taipei_pre_map Visualization
---

In [13]:
# Cell X: Taipei Interactive Risk Map
# total_flood_risk = terrain_flood_risk + river_flood_risk + history_flood_risk

import folium
import geopandas as gpd
import pandas as pd

# ===== 0) Prepare data =====
grid_wgs84 = g.to_crs(epsg=4326).copy()
edges_wgs84 = edges_gdf.to_crs(epsg=4326).copy()
shelters_wgs84 = matched_unique.to_crs(epsg=4326).copy()

# Taipei boundary
town_path = "data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp"
town_gdf = gpd.read_file(town_path)
town_gdf.columns = [str(c).strip().strip("'").strip('"') for c in town_gdf.columns]

if "COUNTYCODE" in town_gdf.columns:
    tp_town = town_gdf[town_gdf["COUNTYCODE"].astype(str).str.contains("63000", na=False)].copy()
elif "COUNTYNAME" in town_gdf.columns:
    tp_town = town_gdf[
        town_gdf["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)
    ].copy()
else:
    tp_town = town_gdf.copy()

tp_town_wgs84 = tp_town.to_crs(epsg=4326)

# ===== 1) Base map =====
minx, miny, maxx, maxy = grid_wgs84.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="cartodbpositron")

# 讓 Grid Tooltip 永遠在最上層，且可以接收滑鼠事件
folium.map.CustomPane(
    "grid_tooltip_pane",
    z_index=900,
    pointer_events=True
).add_to(m)

# ===== 2) Risk grid layers =====
folium.Choropleth(
    geo_data=grid_wgs84[["grid_id", "total_flood_risk", "geometry"]].to_json(),
    data=grid_wgs84[["grid_id", "total_flood_risk"]],
    columns=["grid_id", "total_flood_risk"],
    key_on="feature.properties.grid_id",
    fill_color="OrRd",
    fill_opacity=0.60,
    line_opacity=0.15,
    nan_fill_color="lightgray",
    legend_name="Total Flood Risk",
    name="Total Flood Risk",
    highlight=True,
).add_to(m)

folium.Choropleth(
    geo_data=grid_wgs84[["grid_id", "slope_risk", "geometry"]].to_json(),
    data=grid_wgs84[["grid_id", "slope_risk"]],
    columns=["grid_id", "slope_risk"],
    key_on="feature.properties.grid_id",
    fill_color="YlGnBu",
    fill_opacity=0.60,
    line_opacity=0.15,
    nan_fill_color="lightgray",
    legend_name="Slope Risk",
    name="Slope Risk",
    highlight=True,
    show=False,
).add_to(m)

# ===== 3) Road layers =====
color_map = {
    "expressway": "#e31a1c",
    "arterial": "#1f78b4",
    "bridge": "#ff7f00",
    "residential": "#33a02c",
    "underground": "#6a3d9a",
    "service": "#999999",
}

edges_plot = edges_wgs84.copy()

if "road_type" in edges_plot.columns:
    edges_plot["road_type_std"] = (
        edges_plot["road_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"underground_road": "underground"})
    )
else:
    edges_plot["road_type_std"] = "residential"

for rt, color in color_map.items():
    sub = edges_plot[edges_plot["road_type_std"] == rt].copy()
    if sub.empty:
        continue

    fg = folium.FeatureGroup(name=f"Road: {rt}", show=True)

    folium.GeoJson(
        sub[["geometry", "road_type_std"]].to_json(),
        style_function=lambda x, c=color: {
            "color": c,
            "weight": 1.4,
            "opacity": 0.9
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["road_type_std"],
            aliases=["Road Type"],
            labels=True,
            sticky=False,
        ),
    ).add_to(fg)

    fg.add_to(m)

# ===== 4) Taipei boundary =====
folium.GeoJson(
    tp_town_wgs84[["geometry"]].to_json(),
    name="Taipei Boundary",
    style_function=lambda x: {
        "color": "#000000",
        "weight": 2.2,
        "fillOpacity": 0
    },
).add_to(m)

# ===== 5) Shelters =====
def safe_get(row, col, default="N/A"):
    if col in row.index and pd.notna(row[col]):
        return str(row[col])
    return default

shelter_fg = folium.FeatureGroup(name="Shelters", show=True)

for _, row in shelters_wgs84.iterrows():
    name = safe_get(row, "避難收容處所名稱")
    addr = safe_get(row, "避難收容處所地址")
    cap = safe_get(row, "預計收容人數")
    town = safe_get(row, "縣市及鄉鎮市區")
    hazard = safe_get(row, "適用災害類別")
    indoor = safe_get(row, "室內")
    outdoor = safe_get(row, "室外")
    sid = safe_get(row, "shelter_id")

    popup_html = f"""
    <b>避難收容處所名稱:</b> {name}<br>
    <b>Shelter ID:</b> {sid}<br>
    <b>行政區:</b> {town}<br>
    <b>地址:</b> {addr}<br>
    <b>預計收容人數:</b> {cap}<br>
    <b>適用災害類別:</b> {hazard}<br>
    <b>室內:</b> {indoor}<br>
    <b>室外:</b> {outdoor}
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3.8,
        color="#111111",
        weight=1,
        fill=True,
        fill_color="#ffff33",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=380),
        tooltip=name
    ).add_to(shelter_fg)

shelter_fg.add_to(m)

# ===== 6) Grid Tooltip: must be added last and on top pane =====
folium.GeoJson(
    grid_wgs84[[
        "grid_id",
        "terrain_flood_risk",
        "river_flood_risk",
        "history_flood_risk",
        "total_flood_risk",
        "slope_risk",
        "geometry"
    ]].to_json(),
    name="Grid Tooltip",
    pane="grid_tooltip_pane",
    style_function=lambda x: {
        "color": "#666666",
        "weight": 0.35,
        "opacity": 0.55,
        "fillColor": "#ffffff",
        "fillOpacity": 0.01
    },
    highlight_function=lambda x: {
        "color": "#000000",
        "weight": 1.3,
        "opacity": 1,
        "fillColor": "#ffffff",
        "fillOpacity": 0.08
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "grid_id",
            "terrain_flood_risk",
            "river_flood_risk",
            "history_flood_risk",
            "total_flood_risk",
            "slope_risk"
        ],
        aliases=[
            "Grid ID",
            "Terrain Flood Risk",
            "River Flood Risk",
            "History Flood Risk",
            "Total Flood Risk",
            "Slope Risk"
        ],
        labels=True,
        sticky=False,
    ),
    show=True,
    control=True,
).add_to(m)

# ===== 7) Control & save =====
folium.LayerControl(collapsed=False).add_to(m)

map_path = "output/Taipei_pre_map.html"
m.save(map_path)
print("Saved:", map_path)

Saved: output/Taipei_pre_map.html


In [ ]:
# Cell Final: 將道路依 road_type 編號，並切到 grid 內產生 road_grid_id
# 例：第一條 bridge 穿過 grid 1、grid 2 -> bridge_01_g01, bridge_01_g02

import pandas as pd
import geopandas as gpd
import osmnx as ox

TARGET_CRS = "EPSG:3826"

# 如果 edges_gdf 不存在，就從 G 重新轉出來
if "edges_gdf" not in globals():
    edges_gdf = ox.graph_to_gdfs(G, nodes=False).copy()

# 1) 準備道路資料
roads = edges_gdf.copy().to_crs(TARGET_CRS)

# OSMnx edges 通常 index 是 u, v, key；轉成欄位方便保留
if not isinstance(roads.index, pd.RangeIndex):
    roads = roads.reset_index()

roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

# 確保有 road_type
if "road_type" not in roads.columns:
    roads["road_type"] = "residential"

roads["road_type"] = (
    roads["road_type"]
    .fillna("residential")
    .astype(str)
    .str.strip()
    .str.lower()
)

# 給 road_type 做 ID 用名稱
roads["road_type_id"] = (
    roads["road_type"]
    .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
    .str.strip("_")
)

# 2) 每種 road_type 各自從 1 開始編號
sort_cols = [c for c in ["road_type_id", "u", "v", "key"] if c in roads.columns]
roads = roads.sort_values(sort_cols).reset_index(drop=True)

roads["road_no_in_type"] = roads.groupby("road_type_id").cumcount() + 1

roads["road_base_id"] = roads.apply(
    lambda r: f"{r['road_type_id']}_{int(r['road_no_in_type']):02d}",
    axis=1
)

# 3) 準備 grid
grid_for_road = g[["grid_id", "geometry"]].copy().to_crs(TARGET_CRS)

# 只保留要輸出的道路欄位，避免 GeoJSON 不喜歡 list/dict 欄位
road_cols = [
    c for c in [
        "u", "v", "key",
        "road_type", "road_type_id", "road_no_in_type", "road_base_id",
        "length", "speed_kph", "travel_time",
        "geometry"
    ]
    if c in roads.columns
]

roads_for_overlay = roads[road_cols].copy()

# 4) 將道路切到 grid 範圍內
road_grid_segments = gpd.overlay(
    roads_for_overlay,
    grid_for_road,
    how="intersection",
    keep_geom_type=True
)

# 5) 計算每段在 grid 內的長度
road_grid_segments["segment_length_m"] = road_grid_segments.geometry.length
road_grid_segments = road_grid_segments[
    road_grid_segments["segment_length_m"] > 0
].copy()

# 6) 產生 road_grid_id
road_grid_segments["road_grid_id"] = road_grid_segments.apply(
    lambda r: f"{r['road_base_id']}_g{int(r['grid_id']):02d}",
    axis=1
)

# 7) 調整欄位順序
first_cols = [
    "road_grid_id",
    "road_base_id",
    "road_type",
    "road_no_in_type",
    "grid_id",
    "segment_length_m"
]

other_cols = [
    c for c in road_grid_segments.columns
    if c not in first_cols + ["geometry"]
]

road_grid_segments = road_grid_segments[
    first_cols + other_cols + ["geometry"]
]

# 8) 輸出
out_path = "output/taipei_road_grid_segments.geojson"
road_grid_segments.to_file(out_path, driver="GeoJSON", encoding="utf-8")

print("Saved:", out_path)
print("road-grid segments:", len(road_grid_segments))

display(
    road_grid_segments
    .sort_values(["road_type", "road_no_in_type", "grid_id"])
    [[
        "road_grid_id",
        "road_base_id",
        "road_type",
        "road_no_in_type",
        "grid_id",
        "segment_length_m"
    ]]
    .head(30)
)

Saved: output/taipei_road_grid_segments.geojson
road-grid segments: 101018


,road_grid_id,road_base_id,road_type,road_no_in_type,grid_id,segment_length_m
0,arterial_01_g2025,arterial_01,arterial,1,2025,136.835109
1,arterial_02_g2017,arterial_02,arterial,2,2017,33.029001
2,arterial_02_g2018,arterial_02,arterial,2,2018,24.593818
3,arterial_03_g1834,arterial_03,arterial,3,1834,24.580635
4,arterial_04_g859,arterial_04,arterial,4,859,48.622629
5,arterial_05_g1053,arterial_05,arterial,5,1053,19.616701
6,arterial_06_g1053,arterial_06,arterial,6,1053,101.767588
7,arterial_06_g1129,arterial_06,arterial,6,1129,100.492430
8,arterial_07_g1129,arterial_07,arterial,7,1129,92.222513
9,arterial_08_g1372,arterial_08,arterial,8,1372,31.496862
